# 3. T맵 내비게이션 유입 분석 (보조)

**데이터**: T맵 OD (2025.01~2026.04)
**장점**: 관광지 이름이 직접 나옴 → 방송 소개 관광지 정확히 특정 가능
**지표**: 일별 방문건수, 체류시간, 출발지(원거리 유입 여부)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from config import TMAP_DIR, BROADCAST_DIR, read_csv_auto

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 3.1 T맵 데이터 로드

In [ ]:
# 전체 T맵 데이터 로드 (월별 파일)
dfs = []
for f in sorted(TMAP_DIR.glob('as_tmap_od_*.csv')):
    df = read_csv_auto(f)
    dfs.append(df)
    print(f"{f.name}: {len(df):,}행")

tmap = pd.concat(dfs, ignore_index=True)
print(f"\n총: {len(tmap):,}행")

# 날짜 변환
tmap['date'] = pd.to_datetime(tmap['drv_ymd'], format='%Y%m%d')
tmap['ym'] = tmap['date'].dt.to_period('M')
tmap.head()

## 3.2 방송 촬영 관광지 특정

In [ ]:
# broadcast_locations에서 촬영지 목록
locations = read_csv_auto(BROADCAST_DIR / 'broadcast_locations.csv')
filmed_spots = locations['촬영지명'].dropna().unique().tolist()
print(f"촬영지 {len(filmed_spots)}곳:")
for s in filmed_spots:
    print(f"  {s}")

# T맵 목적지에서 촬영지 매칭
print(f"\nT맵 목적지 유니크: {tmap['dstn_nm'].nunique()}개")

# 키워드 매칭
spot_keywords = {
    '신정호': ['신정호'],
    '현충사': ['현충사'],
    '외암민속마을': ['외암', '민속마을'],
    '온양온천': ['온양온천', '온천'],
    '곡교천': ['곡교천', '은행나무'],
    '피나클랜드': ['피나클'],
    '영인산': ['영인산'],
    '온양온천시장': ['온양.*시장', '온천.*시장'],
    '이순신관광체험센터': ['이순신.*체험', '관광체험'],
    '도고파라다이스': ['도고', '파라다이스'],
    '스파포레': ['스파포레'],
}

def match_spot(name, keywords_dict):
    import re
    for spot, keywords in keywords_dict.items():
        for kw in keywords:
            if re.search(kw, str(name)):
                return spot
    return None

tmap['matched_spot'] = tmap['dstn_nm'].apply(lambda x: match_spot(x, spot_keywords))

matched = tmap[tmap['matched_spot'].notna()]
print(f"\n매칭된 행: {len(matched):,} / {len(tmap):,}")
print(f"\n매칭된 관광지별 건수:")
display(matched.groupby('matched_spot')['vst_cnt'].sum().sort_values(ascending=False))

## 3.3 촬영 관광지 일별 방문 추이

In [ ]:
# 촬영 관광지 일별 방문건수
spot_daily = matched.groupby(['date', 'matched_spot'])['vst_cnt'].sum().unstack('matched_spot').fillna(0)

# 비촬영 관광지 (매칭 안 된 것) 일별 합계
unmatched = tmap[tmap['matched_spot'].isna()]
control_daily = unmatched.groupby('date')['vst_cnt'].sum()
control_daily.name = 'control_기타관광지'

fig, axes = plt.subplots(len(spot_daily.columns), 1, figsize=(15, 3*len(spot_daily.columns)), sharex=True)
if len(spot_daily.columns) == 1:
    axes = [axes]

broadcast_air_dates = {
    '전국노래자랑': pd.Timestamp('2025-06-08'),
    '전현무계획2': pd.Timestamp('2025-11-07'),
    '굿모닝대한민국': pd.Timestamp('2025-11-12'),
    '6시내고향': pd.Timestamp('2025-11-13'),
    '같이삽시다': pd.Timestamp('2025-11-24'),
    '뛰어야산다2': pd.Timestamp('2026-01-12'),
    '황제파워': pd.Timestamp('2026-05-09'),
}

for ax, spot in zip(axes, spot_daily.columns):
    # 7일 이동평균
    raw = spot_daily[spot]
    ma7 = raw.rolling(7, center=True).mean()
    
    ax.fill_between(raw.index, raw.values, alpha=0.2, color='#3498db')
    ax.plot(ma7.index, ma7.values, color='#2c3e50', linewidth=1.5, label='7일 이동평균')
    
    # 방송 시점 표시
    for name, dt in broadcast_air_dates.items():
        if dt >= raw.index.min() and dt <= raw.index.max():
            ax.axvline(dt, color='red', linestyle='--', alpha=0.5)
            ax.text(dt, ax.get_ylim()[1]*0.9, name, fontsize=7, rotation=90, color='red')
    
    ax.set_title(spot)
    ax.set_ylabel('방문건수')

plt.xlabel('날짜')
plt.tight_layout()
plt.show()

## 3.4 출발지 분석: 방송 후 원거리 유입 증가 여부

In [ ]:
# 출발지 시도별 분포: 방송 전 vs 후
def origin_comparison(spot_name, air_date, pre_days=30, post_days=30):
    """특정 관광지의 방송 전후 출발지 분포 비교"""
    spot_data = matched[matched['matched_spot'] == spot_name].copy()
    
    pre_mask = (spot_data['date'] >= air_date - pd.Timedelta(days=pre_days)) & (spot_data['date'] < air_date)
    post_mask = (spot_data['date'] >= air_date) & (spot_data['date'] < air_date + pd.Timedelta(days=post_days))
    
    pre = spot_data[pre_mask].groupby('frst_dptre_ctpv_nm')['vst_cnt'].sum()
    post = spot_data[post_mask].groupby('frst_dptre_ctpv_nm')['vst_cnt'].sum()
    
    comp = pd.DataFrame({'방송전': pre, '방송후': post}).fillna(0)
    comp['변화율'] = ((comp['방송후'] - comp['방송전']) / comp['방송전'].replace(0, np.nan) * 100).round(1)
    comp = comp.sort_values('방송후', ascending=False)
    
    print(f"\n{spot_name} - 방송 전후 출발지 비교 (기준: {air_date.date()})")
    display(comp.head(15))
    
    # 충남 내 vs 충남 외 비율 변화
    local_pre = pre.get('충청남도', 0) / pre.sum() * 100 if pre.sum() > 0 else 0
    local_post = post.get('충청남도', 0) / post.sum() * 100 if post.sum() > 0 else 0
    print(f"충남 비율: {local_pre:.1f}% → {local_post:.1f}%")
    print(f"원거리(충남 외) 비율: {100-local_pre:.1f}% → {100-local_post:.1f}%")
    
    return comp

# 주요 관광지별 실행
for spot in ['신정호', '현충사', '외암민속마을', '곡교천']:
    if spot in matched['matched_spot'].values:
        # 같이삽시다 기준 (가장 시청률 높고 촬영지 많음)
        origin_comparison(spot, pd.Timestamp('2025-11-24'))

## 3.5 체류시간 변화

In [ ]:
# 방송 전후 평균 체류시간 비교
for spot in matched['matched_spot'].dropna().unique():
    spot_data = matched[matched['matched_spot'] == spot]
    
    # 같이삽시다 기준
    air = pd.Timestamp('2025-11-24')
    pre = spot_data[spot_data['date'] < air]['avg_stay_min'].mean()
    post = spot_data[spot_data['date'] >= air]['avg_stay_min'].mean()
    
    if pd.notna(pre) and pd.notna(post):
        change = (post - pre) / pre * 100
        print(f"{spot}: {pre:.0f}분 → {post:.0f}분 ({change:+.1f}%)")